# Inconsistent Dara Entry

In [11]:
import pandas as pd
import numpy as np

import fuzzywuzzy
from fuzzywuzzy import process
import charset_normalizer

#read all the data
professors = pd.read_csv('/Users/caneda/Downloads/pakistan.csv')

professors.head()



,Unnamed: 0,S#,Teacher Name,University Currently Teaching,Department,Province University Located,Designation,Terminal Degree,Graduated from,Country,Year,Area of Specialization/Research Interests,Other Information
0,2,3,Dr. Abdul Basit,University of Balochistan,Computer Science & IT,Balochistan,Assistant Professor,PhD,Asian Institute of Technology,Thailand,NaN,Software Engineering & DBMS,NaN
1,4,5,Dr. Waheed Noor,University of Balochistan,Computer Science & IT,Balochistan,Assistant Professor,PhD,Asian Institute of Technology,Thailand,NaN,DBMS,NaN
2,5,6,Dr. Junaid Baber,University of Balochistan,Computer Science & IT,Balochistan,Assistant Professor,PhD,Asian Institute of Technology,Thailand,NaN,"Information processing, Multimedia mining",NaN
3,6,7,Dr. Maheen Bakhtyar,University of Balochistan,Computer Science & IT,Balochistan,Assistant Professor,PhD,Asian Institute of Technology,Thailand,NaN,"NLP, Information Retrieval, Question Answering...",NaN
4,24,25,Samina Azim,Sardar Bahadur Khan Women's University,Computer Science,Balochistan,Lecturer,BS,Balochistan University of Information Technolo...,Pakistan,2005.0,VLSI Electronics DLD Database,NaN


In [12]:
#get all the unique values in the 'Country' column
countries = professors['Country'].unique()

#sort them alphabetically and then take a closer look
countries.sort()
countries

array([' Germany', ' New Zealand', ' Sweden', ' USA', 'Australia',
       'Austria', 'Canada', 'China', 'Finland', 'France', 'Greece',
       'HongKong', 'Ireland', 'Italy', 'Japan', 'Macau', 'Malaysia',
       'Mauritius', 'Netherland', 'New Zealand', 'Norway', 'Pakistan',
       'Portugal', 'Russian Federation', 'Saudi Arabia', 'Scotland',
       'Singapore', 'South Korea', 'SouthKorea', 'Spain', 'Sweden',
       'Thailand', 'Turkey', 'UK', 'USA', 'USofA', 'Urbana', 'germany'],
      dtype=object)

We can see few mistakes like Germany with germany, because the capitlar letters make different values but for us it should be the same. 

In [13]:
#convert to lower case
countries = [x.lower() for x in countries]

#remove trailing and leading whitespace
countries = [x.strip() for x in countries]  

countries.sort()
countries

['australia',
 'austria',
 'canada',
 'china',
 'finland',
 'france',
 'germany',
 'germany',
 'greece',
 'hongkong',
 'ireland',
 'italy',
 'japan',
 'macau',
 'malaysia',
 'mauritius',
 'netherland',
 'new zealand',
 'new zealand',
 'norway',
 'pakistan',
 'portugal',
 'russian federation',
 'saudi arabia',
 'scotland',
 'singapore',
 'south korea',
 'southkorea',
 'spain',
 'sweden',
 'sweden',
 'thailand',
 'turkey',
 'uk',
 'urbana',
 'usa',
 'usa',
 'usofa']

### Fuzzy matching: 
The process of automatically finding text strings that are very similar to the target string. In general, a string is considered "closer" to another one the fewer characters you need to change if you were transforming one string into another. So "apple" and "snapple" are two changes away from each (add 's' and 'n) while 'in' and 'on' are one changed away. You wont always be able to rely on fuzzy matching 100%, but it will usually end up saving you at least a little time.

In [14]:
# get the top 10 closest matches to "south korea"
matches = fuzzywuzzy.process.extract("south korea", countries, limit=10, scorer=fuzzywuzzy.fuzz.token_sort_ratio)

#take a look at the matches
matches

[('south korea', 100),
 ('southkorea', 48),
 ('saudi arabia', 43),
 ('norway', 35),
 ('ireland', 33),
 ('portugal', 32),
 ('singapore', 30),
 ('netherland', 29),
 ('macau', 25),
 ('usofa', 25)]

In [16]:
#function to replace rows in the provided column of the provided dataframe
#that match the provided string above the provided ratio with the provided string

def replace_matches_in_column(df, column, string_to_match, min_ratio = 47):
    #get a list of unique strings in the provided column
    strings = df[column].unique()
    
    #get the top 10 closest matches to the provided string
    matches = fuzzywuzzy.process.extract(string_to_match, strings, limit=10, scorer=fuzzywuzzy.fuzz.token_sort_ratio)
    
    #only get matches that are above the provided ratio
    close_matches = [matches[0] for matches in matches if matches[1] >= min_ratio]
    
    #replace the close matches with the provided string
    df[column] = df[column].apply(lambda x: string_to_match if x in close_matches else x)
    
    return df

In [17]:
#use the function we just wrote to replace close matches to "south korea" with "south korea"
replace_matches_in_column(df= professors, column= "Country", string_to_match= "south korea")

,Unnamed: 0,S#,Teacher Name,University Currently Teaching,Department,Province University Located,Designation,Terminal Degree,Graduated from,Country,Year,Area of Specialization/Research Interests,Other Information
0,2,3,Dr. Abdul Basit,University of Balochistan,Computer Science & IT,Balochistan,Assistant Professor,PhD,Asian Institute of Technology,Thailand,NaN,Software Engineering & DBMS,NaN
1,4,5,Dr. Waheed Noor,University of Balochistan,Computer Science & IT,Balochistan,Assistant Professor,PhD,Asian Institute of Technology,Thailand,NaN,DBMS,NaN
2,5,6,Dr. Junaid Baber,University of Balochistan,Computer Science & IT,Balochistan,Assistant Professor,PhD,Asian Institute of Technology,Thailand,NaN,"Information processing, Multimedia mining",NaN
3,6,7,Dr. Maheen Bakhtyar,University of Balochistan,Computer Science & IT,Balochistan,Assistant Professor,PhD,Asian Institute of Technology,Thailand,NaN,"NLP, Information Retrieval, Question Answering...",NaN
4,24,25,Samina Azim,Sardar Bahadur Khan Women's University,Computer Science,Balochistan,Lecturer,BS,Balochistan University of Information Technolo...,Pakistan,2005.0,VLSI Electronics DLD Database,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1137,1974,1975,Dr. Ahmar Rashid,Ghulam Ishaq Khan Institute,Computer Science and Engineering,KPK,Associate Professor,PhD,JNU,south korea,NaN,"Electrical Impedance Tomography, Inverse algor...",NaN
1138,1975,1976,Dr. Fawad Hussain,Ghulam Ishaq Khan Institute,Computer Science and Engineering,KPK,Associate Professor,PhD,Grenoble,France,NaN,"Machine Learning, Big Data Anaysis, Data Minin...",NaN
1139,1977,1978,Dr. Rashad M Jillani,Ghulam Ishaq Khan Institute,Computer Science and Engineering,KPK,Assistant Professor,PhD,Florida Atlantic University,USA,2012.0,"Digital Multimedia Systems, Video Compression ...",NaN
1140,1979,1980,Dr. Shahabuddin Ansari,Ghulam Ishaq Khan Institute,Computer Science and Engineering,KPK,Assistant Professor,PhD,Ghulam Ishaq Khan Institute of Science and Tec...,Pakistan,NaN,"Medical Image Processing and Analysis, Digital...",NaN


In [18]:
#get all the unique values in the 'Country' column
countries = professors['Country'].unique()

#sort them alphabetically and then take a closer look
countries.sort()
countries

array([' Germany', ' New Zealand', ' Sweden', ' USA', 'Australia',
       'Austria', 'Canada', 'China', 'Finland', 'France', 'Greece',
       'HongKong', 'Ireland', 'Italy', 'Japan', 'Macau', 'Malaysia',
       'Mauritius', 'Netherland', 'New Zealand', 'Norway', 'Pakistan',
       'Portugal', 'Russian Federation', 'Saudi Arabia', 'Scotland',
       'Singapore', 'Spain', 'Sweden', 'Thailand', 'Turkey', 'UK', 'USA',
       'USofA', 'Urbana', 'germany', 'south korea'], dtype=object)